<a href="https://colab.research.google.com/github/juandagmz/Proyecto_final/blob/main/Proyecto_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Análisis Automático de Reseñas de Productos
**Asignatura:** Inteligencia Artificial — Ingeniería de Sistemas  
**Objetivo:** Clasificar reseñas como positivas o negativas y exponer el modelo como API con ngrok.

---
## Contenido
1. Instalación de dependencias
2. Dataset y preprocesamiento
3. Entrenamiento del modelo (TF-IDF + Logistic Regression)
4. Evaluación: accuracy, matriz de confusión, ejemplos comentados
5. API con Flask + ngrok (para conectar con Make)

## 1.  Instalación de dependencias

In [ ]:
!pip install flask flask-ngrok pyngrok scikit-learn pandas matplotlib seaborn nltk -q
import nltk
nltk.download('stopwords')
nltk.download('punkt')
print('✅ Dependencias instaladas')

## 2. Dataset y Preprocesamiento

Usamos un dataset propio simulado

In [ ]:
import pandas as pd
import numpy as np

data = {
    'texto': [
        # Positivas
        'Excelente producto, llegó rápido y en perfecto estado',
        'Muy buena calidad, lo recomiendo totalmente',
        'Quedé encantado, funciona perfecto y el precio es justo',
        'Increíble, superó mis expectativas, lo compraría de nuevo',
        'Producto de buena calidad, entrega rápida, muy satisfecho',
        'Me gustó mucho, es exactamente lo que necesitaba',
        'Excelente relación calidad-precio, muy recomendado',
        'Llegó antes de lo esperado y en perfectas condiciones',
        'Muy buen producto, mi familia quedó feliz',
        'Todo perfecto, el vendedor muy atento y el producto genial',
        'Funciona de maravilla, muy contento con la compra',
        'Buenísimo, la calidad es notablemente superior al precio',
        'Hermoso producto, llegó bien empacado y sin daños',
        'Muy satisfecho, lo usé el mismo día que llegó',
        'Superó todas mis expectativas, definitivamente lo recomiendo',
        # Negativas
        'Pésimo producto, se dañó al tercer día de uso',
        'Muy mala calidad, no lo recomiendo para nada',
        'Llegó roto y el vendedor no responde, terrible experiencia',
        'Decepcionante, no es lo que mostraba la foto',
        'Tarda demasiado en llegar y el producto es de muy mala calidad',
        'No sirve, una pérdida de dinero total',
        'Muy mal servicio, el producto llegó incompleto',
        'Pésima atención al cliente, el producto falló a la semana',
        'No funciona como dice la descripción, me siento estafado',
        'Horrible, el material es plástico barato que se rompe fácil',
        'Tardó tres semanas y llegó dañado, inaceptable',
        'Producto defectuoso, intenté devolverlo y no me atienden',
        'Malísimo, el olor es insoportable y los colores no son fieles',
        'Una basura, no dura nada y el soporte técnico no existe',
        'Muy disappointing, el producto no cumple ninguna de sus promesas',
    ],
    'etiqueta': [1]*15 + [0]*15  # 1=positivo, 0=negativo
}

df = pd.DataFrame(data)
print(f'Dataset cargado: {len(df)} reseñas')
print(f'Positivas: {sum(df.etiqueta==1)} | Negativas: {sum(df.etiqueta==0)}')
df.head()

In [ ]:
import re
from nltk.corpus import stopwords

stop_words_es = set(stopwords.words('spanish'))

def limpiar_texto(texto):
    """Limpia y normaliza el texto de una reseña."""
    texto = texto.lower()                              # minúsculas
    texto = re.sub(r'[^a-záéíóúüñ\s]', '', texto)     # quitar signos
    texto = re.sub(r'\s+', ' ', texto).strip()         # espacios extra
    tokens = texto.split()
    tokens = [t for t in tokens if t not in stop_words_es]  # stopwords
    return ' '.join(tokens)

df['texto_limpio'] = df['texto'].apply(limpiar_texto)
print('✅ Preprocesamiento completado')
df[['texto', 'texto_limpio']].head(3)

## 3. Entrenamiento del Modelo

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

X = df['texto_limpio']
y = df['etiqueta']

# Como el dataset es pequeño (ya que es simulado) usamos todo para la demostración
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Pipeline: TF-IDF → Logistic Regression
modelo = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1, 2),
        max_features=5000,
        sublinear_tf=True
    )),
    ('clf', LogisticRegression(max_iter=1000, random_state=42))
])

modelo.fit(X_train, y_train)
print('✅ Modelo entrenado')
print(f'Tamaño entrenamiento: {len(X_train)} | Tamaño prueba: {len(X_test)}')

## Evaluación del Modelo

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

y_pred = modelo.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f'🎯 Accuracy: {acc:.2%}')
print()
print('📊 Reporte de clasificación:')
print(classification_report(y_test, y_pred, target_names=['Negativa', 'Positiva']))

In [ ]:
# Matriz de confusión
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negativa', 'Positiva'],
            yticklabels=['Negativa', 'Positiva'], ax=ax)
ax.set_xlabel('Predicción')
ax.set_ylabel('Real')
ax.set_title('Matriz de Confusión')
plt.tight_layout()
plt.show()

print(f"""
Interpretación:
  ✅ Verdaderos Negativos (TN): {cm[0][0]}  — negativas correctamente identificadas
  ❌ Falsos Positivos  (FP): {cm[0][1]}  — negativas clasificadas como positivas
  ❌ Falsos Negativos  (FN): {cm[1][0]}  — positivas clasificadas como negativas
  ✅ Verdaderos Positivos (TP): {cm[1][1]}  — positivas correctamente identificadas
""")

## Ejemplos comentados

In [ ]:
def clasificar(texto):
    limpio = limpiar_texto(texto)
    pred = modelo.predict([limpio])[0]
    proba = modelo.predict_proba([limpio])[0]
    etiqueta = '✅ POSITIVA' if pred == 1 else '❌ NEGATIVA'
    confianza = max(proba)
    return etiqueta, confianza

ejemplos = [
    {
        'tipo': '1️⃣ Bien clasificado (claramente positiva)',
        'texto': 'El producto es increíble, llegó rápido y funciona perfecto, lo recomiendo',
        'esperado': 'POSITIVA'
    },
    {
        'tipo': '2️⃣ Mal clasificado (negativa con lenguaje ambiguo)',
        'texto': 'No esperaba que fuera tan bueno... me sorprendió para mal',
        'esperado': 'NEGATIVA'
    },
    {
        'tipo': '3️⃣ Dudoso (sarcasmo / ironía)',
        'texto': 'Claro que "funciona perfecto" si por perfecto entiendes que se rompe al día',
        'esperado': 'NEGATIVA'
    },
]

for ej in ejemplos:
    etiqueta, confianza = clasificar(ej['texto'])
    correcto = '✔️' if ej['esperado'] in etiqueta else '✘ ERROR'
    print(f"{ej['tipo']}")
    print(f"  Texto: \"{ej['texto']}\"")
    print(f"  Predicción: {etiqueta} (confianza: {confianza:.0%}) {correcto}")
    print()

## API con Flask + ngrok


In [ ]:
from pyngrok import ngrok

# 🔑 REEMPLAZA CON TU TOKEN DE NGROK (gratis en ngrok.com)
NGROK_TOKEN = '3EYUZCWSnEwVHKiLJMkQdMXf0r7_5jAdXuLcSudbqZanRfU8f'
ngrok.set_auth_token(NGROK_TOKEN)
print('Token de ngrok configurado')

In [ ]:
from flask import Flask, request, jsonify
import threading

app = Flask(__name__)

@app.route('/clasificar', methods=['POST'])
def clasificar_resena():
    """
    Endpoint para clasificar una reseña.
    Body JSON esperado: { "resena": "texto de la reseña" }
    Respuesta: { "sentimiento": "positiva" | "negativa", "confianza": 0.95 }
    """
    try:
        data = request.get_json()
        resena = data.get('resena', '')

        if not resena:
            return jsonify({'error': 'Campo "resena" requerido'}), 400

        limpio = limpiar_texto(resena)
        pred = modelo.predict([limpio])[0]
        proba = modelo.predict_proba([limpio])[0]

        resultado = {
            'sentimiento': 'positiva' if pred == 1 else 'negativa',
            'confianza': round(float(max(proba)), 4),
            'resena_recibida': resena
        }
        return jsonify(resultado), 200

    except Exception as e:
        return jsonify({'error': str(e)}), 500


@app.route('/salud', methods=['GET'])
def salud():
    return jsonify({'estado': 'ok', 'mensaje': 'API de análisis de sentimiento activa'}), 200


# Iniciar ngrok y Flask en un hilo separado
public_url = ngrok.connect(5000)
print(f'\n🚀 API pública disponible en: {public_url}')
print(f'📌 Endpoint para Make: {public_url}/clasificar')
print(f'📌 Health check:       {public_url}/salud')
print('\n⚠️  Deja esta celda corriendo mientras uses Make.')
print('   Para detener: Kernel → Interrupt')

app.run(port=5000)